# SPOOL — photon-efficient quantitative FLIM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wonsang7/SPOOL/blob/main/SPOOL_demo.ipynb)

**SPOOL** (Spatially Pooled Optical Observation Likelihood) is a training-free Poisson inverse framework that explicitly uses the microscope point-spread function (PSF).

This demo simulates a few-photon FLIM acquisition, reconstructs the same photon counts with pixel-wise Poisson MLE and SPOOL, compares lifetime RMSE, and visualizes the result.

> **Quick start:** choose a GPU runtime and run all cells. A T4 GPU is recommended.


## 1. Setup


In [ ]:
import sys, shutil, subprocess, time, io
from pathlib import Path

REPO_URL = 'https://github.com/Wonsang7/SPOOL.git'
REPO_DIR = Path('/content/SPOOL_repo')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','-q',REPO_URL,str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_DIR/'requirements.txt')], check=True)
repo_path = str(REPO_DIR.resolve())
sys.path.insert(0, repo_path)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import flim_sim_ncpca as fs
import run_multiemitter_benchmark as bm

commit = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'], text=True).strip()
print(f'✓ SPOOL commit: {commit}')
print(f'✓ Compute backend: {bm.BACKEND}')


## 2. Build the lifetime dictionary and PSF operator

The reconstruction uses the manuscript's lifetime-agnostic 11-component dictionary and explicit PSF model.


In [ ]:
bm.D_REC = bm.build_decay_basis(bm.TAU_REC)
D_d = bm.to_dev(bm.D_REC)
Dsum_d = bm.to_dev(bm.D_REC.sum(axis=1))
H = W = fs.IMAGE_SIZE
o = bm.make_otf(np.asarray(fs.PSF), (H, W))
if bm.USE_TORCH:
    import torch
    ones = torch.ones((bm.K_REC,H,W), dtype=torch.float64, device=bm.DEVICE)
    C = bm.conv_batch(ones,o) * Dsum_d.reshape(bm.K_REC,1,1)
else:
    ones = np.ones((bm.K_REC,H,W))
    C = bm.conv_batch(ones,o) * np.asarray(Dsum_d).reshape(bm.K_REC,1,1)
print(f'✓ Dictionary: K={bm.K_REC}, {bm.TAU_REC[0]:.1f}–{bm.TAU_REC[-1]:.1f} ns')
print(f'✓ PSF: {fs.PSF_FWHM_NM:.0f} nm FWHM at {fs.PIXEL_SIZE_NM} nm/pixel')


## 3. Simulate a few-photon acquisition


In [ ]:
#@title Choose photon budget
photons_per_bead = 200  #@param [50, 100, 200, 400, 800] {type:'raw'}
cfg = {'n_species0': None}
rng = np.random.default_rng(2000)
bg = bm.background_for_level(photons_per_bead, cfg)
fs.BACKGROUND_RATE = bg
A_true, coords = fs.generate_scene(bm.N_BEADS, photons_per_bead, rng, n_species0=None)
species = np.array([np.argmax(A_true[:,y,x]) for (x,y) in coords])
tau_true = np.asarray(fs.TAUS_NS)[species]
Y, Lam = fs.simulate_measurement(A_true, rng)
fg_mean, _ = bm.photon_budget(Lam, coords)
print(f'✓ {len(coords)} emitters; {fg_mean:.1f} detected photons per foreground pixel')


## 4. Reconstruct with pixel-wise MLE and SPOOL


In [ ]:
t0 = time.time()
with bm.reconstruction_dictionary():
    Y_d = bm.to_dev(Y)
    A_mle = bm.to_np(bm.mle_gpu(Y_d, D_d, Dsum_d, bg))
    A_sp = bm.to_np(bm.joint_gpu(Y_d, D_d, o, C, bg))
elapsed = time.time()-t0
tau_mle = bm.lifetime_at(A_mle, coords, bm.TAU_REC)
tau_sp = bm.lifetime_at(A_sp, coords, bm.TAU_REC)
rmse_mle = float(np.sqrt(np.mean((tau_mle-tau_true)**2)))
rmse_sp = float(np.sqrt(np.mean((tau_sp-tau_true)**2)))
print(f'✓ Reconstruction time: {elapsed:.2f} s')
print(f'Pixel-wise MLE RMSE: {rmse_mle:.3f} ns')
print(f'SPOOL RMSE:          {rmse_sp:.3f} ns')
print(f'RMSE improvement:    {rmse_mle/rmse_sp:.2f}×')

cmap = plt.cm.rainbow.copy(); cmap.set_bad('black')
panels = [('Ground truth', bm.masked_map(bm.tau_map(A_true,np.asarray(fs.TAUS_NS)),A_true.sum(0),frac=0.5)), ('Detected photons',None), ('Pixel-wise MLE',bm.masked_map(bm.tau_map(A_mle,bm.TAU_REC),A_mle.sum(0),frac=0.05)), ('SPOOL',bm.masked_map(bm.tau_map(A_sp,bm.TAU_REC),A_sp.sum(0),frac=0.05))]
fig, axes = plt.subplots(1,4,figsize=(16,4.2))
for ax,(title,m) in zip(axes,panels):
    if m is None:
        im=ax.imshow(Y.sum(2),cmap='magma'); fig.colorbar(im,ax=ax,fraction=0.046,label='photons')
    else:
        im=ax.imshow(m,cmap=cmap,vmin=2,vmax=4); fig.colorbar(im,ax=ax,fraction=0.046,label='lifetime (ns)')
    ax.set_title(title); ax.axis('off')
fig.suptitle(f'{fg_mean:.1f} photons per foreground pixel',y=1.03); fig.tight_layout(); plt.show()


## 5. Optional photon-efficiency sweep

Enable the checkbox to run a short 3-level × 3-trial comparison.


In [ ]:
#@title Run mini sweep?
RUN_PHOTON_SWEEP = False  #@param {type:'boolean'}
if not RUN_PHOTON_SWEEP:
    print('Mini sweep skipped.')
else:
    levels=[50,200,800]; xs=[]; mle_mu=[]; spool_mu=[]
    for pl in levels:
        bg_s=bm.background_for_level(pl,cfg); fs.BACKGROUND_RATE=bg_s; r_m=[]; r_s=[]; fg=[]
        for trial in range(3):
            rr=np.random.default_rng(2000+trial); At,crd=fs.generate_scene(bm.N_BEADS,pl,rr,n_species0=None); sp=np.array([np.argmax(At[:,y,x]) for (x,y) in crd]); tt=np.asarray(fs.TAUS_NS)[sp]; Yt,Lt=fs.simulate_measurement(At,rr); fg.append(bm.photon_budget(Lt,crd)[0])
            with bm.reconstruction_dictionary():
                Yd=bm.to_dev(Yt); Am=bm.to_np(bm.mle_gpu(Yd,D_d,Dsum_d,bg_s)); As=bm.to_np(bm.joint_gpu(Yd,D_d,o,C,bg_s))
            r_m.append(np.sqrt(np.mean((bm.lifetime_at(Am,crd,bm.TAU_REC)-tt)**2))); r_s.append(np.sqrt(np.mean((bm.lifetime_at(As,crd,bm.TAU_REC)-tt)**2)))
        xs.append(np.mean(fg)); mle_mu.append(np.mean(r_m)); spool_mu.append(np.mean(r_s)); print(f'{xs[-1]:.1f} ph/px | MLE {mle_mu[-1]:.3f} ns | SPOOL {spool_mu[-1]:.3f} ns')
    plt.figure(figsize=(5.5,4)); plt.plot(xs,mle_mu,'o-',label='Pixel-wise MLE'); plt.plot(xs,spool_mu,'o-',label='SPOOL'); plt.xscale('log'); plt.xlabel('mean photons per foreground pixel'); plt.ylabel('lifetime RMSE (ns)'); plt.legend(); plt.tight_layout(); plt.show()


## Going further

Full benchmark: `python run_multiemitter_benchmark.py`  
Experimental drivers: `run_bead_microtubule.py`, `run_dual_labeled_flim.py`, and `run_hyperspectral.py`.
